# Salud relativa: del registro al modelo y del modelo al índice
## Cuaderno para presentar y reproducir el experimento de urgencias

**Pregunta:** ¿qué capacidad de urgencias podemos estimar cuando falta su registro municipal, y cómo cambia el puntaje de Salud al usarla?

Este notebook usa exclusivamente datos y resultados del repositorio `Practicantepotencia/tablero-terremoto`, congelados en el commit **a350ca6591174542e76b08d108e4d4e15c629b4c**. No modifica el tablero ni sus archivos.

**Recorrido:** fuentes → cobertura → variables → Poisson → comparación de cinco modelos → validación geográfica → gráficos → faltantes → puntaje de Salud → caso Atrato → límites para la presentación.

- **Lo que se predice:** consultorios de urgencias registrados, NO consultas atendidas, IPS, camas ni daños.
- **Lo que se conserva:** todo denominador observado; todo daño original.
- **Lo que no se puede afirmar:** que una predicción confirme la existencia del servicio o su funcionamiento actual.

**Estado de esta entrega:** las celdas Python se entregan sin ejecutar: el entorno de autoría bloqueó Python. Los resultados históricos citados proceden de la corrida JavaScript guardada en el repositorio; las comprobaciones de abajo contrastan la nueva ejecución Python con ella. No hay salidas de Python simuladas.

**Cómo usarlo:** Jupyter, VS Code con kernel Python o Google Colab; ejecutar de arriba hacia abajo. La selección anidada ajusta muchos modelos y puede tardar varios minutos. Antes de presentar, ejecutar todo y guardar el notebook con sus salidas. Las descargas suman aproximadamente 27 MB, principalmente el tablero con los datos originales.


## 1. Preparar el entorno
El entrenamiento se muestra en Python/NumPy, como traducción explícita del algoritmo original de `scripts/ml_salud.cjs`. No se sustituye el boosting original por otro algoritmo con el mismo nombre.

Se usa scikit-learn **sólo como contraste independiente del ajuste Poisson**. Si no está instalado, esa comprobación se omite con un mensaje; el resto puede ejecutarse.

Si faltan dependencias, descomenta y ejecuta la primera línea de la celda siguiente. En una instalación local también puedes usar el archivo `notebooks/requirements-salud.txt`.


In [ ]:
# %pip install "numpy>=1.24,<3" "pandas>=2,<4" "matplotlib>=3.7,<4" "scikit-learn>=1.3,<2"
import sys, json, hashlib, io, re, math, platform, warnings
from pathlib import Path
from urllib.request import Request, urlopen
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
AZUL, NARANJA = "#12659a", "#d66a3a"
display(pd.Series({"Python": sys.version.split()[0], "NumPy": np.__version__,
                   "pandas": pd.__version__, "plataforma": platform.platform()}))


## 2. Cargar una versión identificable, no «lo último» sin control
Se intenta leer del repositorio local. El archivo debe coincidir con el **hash Git del archivo de la versión congelada**. Si no coincide, se informa y se descarga esa versión exacta. Si no hay acceso de red, se detiene: no se usa silenciosamente otro corte.

Los JSON de entrada ya contienen agregados municipales, sin contactos personales. Este cuaderno parte de esos agregados; no vuelve a descargar ni auditar nominalmente el registro nacional de prestadores.

**Para decir en la presentación:** «Podemos identificar exactamente los archivos con los que se produjo este resultado».


In [ ]:
REPO = "Practicantepotencia/tablero-terremoto"
COMMIT = "a350ca6591174542e76b08d108e4d4e15c629b4c"
RAW = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/"
GIT_BLOBS = {
    "experimentos/ml_salud/entrada.json": "989bac0dcbf1e89d778c7494adc0cfb4e66165b4",
    "experimentos/ml_salud/resultados.json": "0d0dabd32ee5f6fdfac59bbb7a1f656b945d9e2f",
    "experimentos/ml_salud/validacion.csv": "d14913c0608b6898e5d21ef362939d1ed75a6884",
    "experimentos/ml_salud/impacto_aplicacion_urgencias.json": "95efc6f0cae6bf0deaacb163dd9ab0bdc5fb8b3d",
    "index.html": "98592fadffd7fb9d7b821b7420c8948be6c4aef1"
}
_cache = {}
trazabilidad = []

def git_blob_sha(data):
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()

def leer_version(path):
    if path in _cache:
        return _cache[path]
    esperado = GIT_BLOBS[path]
    contenido, procedencia = None, None
    for raiz in [Path.cwd(), *Path.cwd().parents]:
        local = raiz / path
        if local.is_file():
            candidato = local.read_bytes()
            if git_blob_sha(candidato) == esperado:
                contenido, procedencia = candidato, str(local)
                break
            print(f"Se omite copia local distinta del corte: {local}")
            break
    if contenido is None:
        url = RAW + path
        try:
            with urlopen(Request(url, headers={"User-Agent": "Notebook-Salud-Reproducible"}), timeout=120) as respuesta:
                contenido = respuesta.read()
            procedencia = url
        except Exception as exc:
            raise RuntimeError(
                f"No se pudo cargar {path}. Se necesita acceso al repositorio, "
                f"o una copia local exacta del commit {COMMIT}."
            ) from exc
    obtenido = git_blob_sha(contenido)
    if obtenido != esperado:
        raise ValueError(f"Integridad incorrecta: {path}: {obtenido} != {esperado}")
    _cache[path] = contenido
    trazabilidad.append({"archivo": path, "blob_git": obtenido, "bytes": len(contenido),
                         "origen": procedencia})
    return contenido

entrada = json.loads(leer_version("experimentos/ml_salud/entrada.json"))
archivo_resultados = json.loads(leer_version("experimentos/ml_salud/resultados.json"))
historico = next(r for r in archivo_resultados["results"] if r["target"]["id"] == "urgencias")
validacion_archivada = pd.read_csv(
    io.BytesIO(leer_version("experimentos/ml_salud/validacion.csv")), dtype={"codigo": str}
)
display(pd.DataFrame(trazabilidad))


## 3. Qué mide cada fuente y de qué año es

| Información | Fuente de origen | Referencia |
|---|---|---|
| Consultorios de urgencias, consulta externa y camas generales | Ministerio de Salud / REPS | 5 de noviembre de 2022 |
| Población total municipal | Proyecciones DANE | 2026 |
| IPM municipal | DANE, fuente censal | 2018 |
| Centros de salud afectados | Inventario PNUD del tablero | Captura del 11 de septiembre de 2026 |

El extracto REPS se conservó a través de un espejo inmutable; la URL y los hashes están en la entrada. **Fecha de publicación del archivo no equivale a fecha del registro.** Las fechas de captura del tablero tampoco prueban la fecha de cada observación de daño.

Esta combinación temporal permite un ensayo exploratorio; no acredita capacidad operativa en 2026.


In [ ]:
display(pd.DataFrame([
    {"variable": "Capacidad sanitaria", "referencia": entrada["target_date"],
     "fuente": entrada["sources"]["capacity"]["url"],
     "extracto": entrada["sources"]["capacity"]["mirror_url"]},
    {"variable": "Población", "referencia": entrada["population_year"],
     "fuente": entrada["sources"]["population"]["url"]},
    {"variable": "IPM", "referencia": entrada["ipm_year"],
     "fuente": entrada["sources"]["ipm"]["url"]},
]))

df = pd.DataFrame(entrada["rows"])
df["code"] = df["code"].astype(str).str.zfill(5)
for j, nombre in enumerate(["consulta_externa", "urgencias", "camas_generales"]):
    df[nombre] = df["capacity_2022"].apply(lambda v: v[j]).astype(float)

assert len(df) == 1122 and df["code"].is_unique
assert df["code"].str.fullmatch(r"\d{5}").all()
assert np.isfinite(df["population_2026"]).all() and df["population_2026"].gt(0).all()
for c in ["consulta_externa", "urgencias", "camas_generales"]:
    assert df[c].dropna().gt(0).all(), f"Revisar ceros/negativos en {c}"
display(df[["code", "municipality", "department", "population_2026",
            "ipm_2018", "consulta_externa", "urgencias", "camas_generales"]].head(10))


## 4. Cobertura antes de entrenar
Un **faltante no es cero**. No entrenamos con etiquetas inventadas ni repetimos el mismo municipio por cada captura diaria.

En urgencias hay 901 etiquetas positivas y 221 ausencias de registro. Aprender sólo con positivos no permite aprender cuándo el servicio realmente no existe.


In [ ]:
cobertura = pd.DataFrame([
    {"denominador": c, "observados_positivos": int(df[c].notna().sum()),
     "faltantes": int(df[c].isna().sum()), "ceros_observados": int(df[c].eq(0).sum())}
    for c in ["consulta_externa", "urgencias", "camas_generales"]
])
display(cobertura)
observados = df.loc[df["urgencias"].notna()].copy().reset_index(drop=True)
faltantes = df.loc[df["urgencias"].isna()].copy().reset_index(drop=True)
assert len(observados) == 901 and len(faltantes) == 221
assert observados["population_2026"].lt(50000).sum() == 750

fig, ax = plt.subplots()
ax.bar(cobertura["denominador"], cobertura["observados_positivos"], color=AZUL, label="Registro positivo")
ax.bar(cobertura["denominador"], cobertura["faltantes"],
       bottom=cobertura["observados_positivos"], color="#dbe4ea", label="Sin registro utilizable")
ax.set(ylabel="Municipios", title="Cobertura nacional de capacidad sanitaria")
ax.legend(); plt.tight_layout(); plt.show()


## 5. Construir los predictores sin filtrar información de validación
Para urgencias usamos siete columnas:

1. \(\log(1+\text{población})\).
2. IPM.
3. Bandera de IPM ausente.
4. \(\log(1+\text{consulta externa})\), o un 0 técnico si falta.
5. Bandera de consulta externa ausente.
6. \(\log(1+\text{camas generales})\), o un 0 técnico si falta.
7. Bandera de camas ausentes.

El 0 técnico con bandera **no es una observación de capacidad cero**. Las medianas, medias y desviaciones se calculan de nuevo **sólo con cada conjunto de entrenamiento**.

No entran daños, puntajes PNUD/UNGRD, el índice final ni el propio valor de urgencias. Tampoco usamos otras capacidades predichas para predecir ésta.


In [ ]:
NOMBRES_X = [
    "log1p(población 2026)", "IPM 2018", "IPM faltante",
    "log1p(consulta externa 2022)", "consulta externa faltante",
    "log1p(camas generales 2022)", "camas generales faltantes"
]

def variables(frame, enriquecido=True):
    columnas = [
        np.log1p(frame["population_2026"].to_numpy(dtype=float)),
        frame["ipm_2018"].to_numpy(dtype=float),
        frame["ipm_2018"].isna().to_numpy(dtype=float),
    ]
    if enriquecido:
        for c in ["consulta_externa", "camas_generales"]:
            columnas.extend([
                np.log1p(frame[c].fillna(0).to_numpy(dtype=float)),
                frame[c].isna().to_numpy(dtype=float),
            ])
    return np.column_stack(columnas)

def ajustar_preproceso(frame, enriquecido=True):
    X = variables(frame, enriquecido)
    medianas = np.array([
        np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.
        for col in X.T
    ])
    completo = np.where(np.isfinite(X), X, medianas)
    medias = completo.mean(axis=0)
    desv = completo.std(axis=0, ddof=0)
    desv = np.where(desv == 0, 1., desv)
    return {"med": medianas, "avg": medias, "sd": desv}

def transformar(frame, pre, enriquecido=True):
    X = variables(frame, enriquecido)
    return (np.where(np.isfinite(X), X, pre["med"]) - pre["avg"]) / pre["sd"]

display(pd.DataFrame(variables(observados), columns=NOMBRES_X).head())


## 6. Entrenar Poisson: qué función se minimiza
Para \(z_i\), los predictores estandarizados del municipio \(i\):

\[
\eta_i=\beta_0+z_i^\top\beta,\qquad \mu_i=\exp(\eta_i)
\]

\[
L(\beta_0,\beta)=\frac1n\sum_i\left[\exp(\eta_i)-y_i\eta_i\right]
+\frac{\alpha}{2}\sum_j\beta_j^2,\qquad \alpha=1
\]

El término final es la regularización L2. El intercepto no se penaliza. Se omiten constantes que no cambian el ajuste.

La predicción publicada se recorta después:

\[
\widehat B_i=\max(1,\mu_i).
\]

**Importante:** el piso 1 no forma parte de la optimización; es una restricción posterior del ensayo de positivos. No hemos ajustado un modelo que distinga ceros reales ni una distribución Poisson truncada formal.

La celda muestra Newton con búsqueda de paso, como en el código original. NumPy resuelve el sistema lineal.


In [ ]:
def ajustar_poisson(X, y, alpha=1.):
    y = np.asarray(y, dtype=float)
    X1 = np.column_stack([np.ones(len(y)), X])
    n, p = X1.shape
    beta = np.zeros(p)
    beta[0] = np.log(y.mean())
    penalizacion = np.r_[0., np.full(p - 1, alpha)]

    def perdida(b):
        eta = X1 @ b
        if not np.isfinite(eta).all() or np.max(eta) > 30:
            return np.inf
        return np.mean(np.exp(eta) - y * eta) + alpha * np.sum(b[1:]**2) / 2

    convergio = False
    for it in range(80):
        mu = np.exp(X1 @ beta)
        gradiente = X1.T @ (mu - y) / n + penalizacion * beta
        hessiano = (X1.T * mu) @ X1 / n + np.diag(penalizacion)
        if np.max(np.abs(gradiente)) < 1e-7:
            convergio = True
            break
        paso = np.linalg.solve(hessiano, gradiente)
        previo, tasa, aceptado = perdida(beta), 1., False
        for _ in range(35):
            candidato = beta - tasa * paso
            if perdida(candidato) <= previo - 1e-4 * tasa * float(gradiente @ paso):
                beta, aceptado = candidato, True
                break
            tasa /= 2
        if not aceptado:
            convergio = np.max(np.abs(paso)) < 1e-6
            break
        if np.max(np.abs(tasa * paso)) < 1e-8:
            convergio = True
            break
    return {"kind": "poisson", "beta": beta, "converged": bool(convergio),
            "iterations": it + 1, "alpha": alpha}

# Prueba analítica: sin información predictiva y con y constante, se recupera su media.
prueba = ajustar_poisson(np.zeros((40, 2)), np.full(40, 3.))
assert prueba["converged"]
np.testing.assert_allclose(np.exp(prueba["beta"][0]), 3., rtol=1e-8)
np.testing.assert_allclose(prueba["beta"][1:], 0., atol=1e-8)
print("Prueba analítica de Poisson: correcta.")


## 7. Los modelos alternativos
Se compararon cinco candidatos, no sólo Poisson:

| Candidato | Predictores |
|---|---|
| Mediana por cinco grupos de población | Población |
| Poisson demográfico | Población e IPM |
| Poisson con capacidades | Población, IPM y las otras capacidades |
| Boosting demográfico | Población e IPM |
| Boosting con capacidades | Población, IPM y las otras capacidades |

El boosting original tiene 160 árboles, profundidad 2, mínimo 20 observaciones por hoja, tasa 0,05 y hasta 24 cortes por variable. Ajusta errores cuadrados de \(\log(1+y)\), **no devianza Poisson**. La siguiente celda traduce ese algoritmo; no utiliza los parámetros por defecto de otra biblioteca.

**Para enseñar:** los árboles corrigen gradualmente el error del conjunto anterior; Poisson aprende una fórmula exponencial única.


In [ ]:
def valor_arbol(arbol, X):
    if "feature" not in arbol:
        return np.full(len(X), arbol["value"])
    lado = X[:, arbol["feature"]] <= arbol["threshold"]
    salida = np.empty(len(X))
    salida[lado] = valor_arbol(arbol["left"], X[lado])
    salida[~lado] = valor_arbol(arbol["right"], X[~lado])
    return salida

def ajustar_boosting(X, y, rounds=160, depth=2, min_leaf=20, rate=.05, bins=24):
    n, p = X.shape
    z = np.log1p(y)
    base = float(z.mean())
    pred = np.full(n, base)
    arboles, cortes = [], []
    for j in range(p):
        vals = np.unique(X[:, j])
        if len(vals) <= bins + 1:
            puntos = (vals[:-1] + vals[1:]) / 2
        else:
            ks = [int(i * len(vals) / (bins + 1)) for i in range(1, bins + 1)]
            puntos = np.array([(vals[k-1] + vals[k]) / 2 for k in ks])
        cortes.append(np.unique(puntos))
    binX = np.column_stack([np.searchsorted(cortes[j], X[:, j], side="left") for j in range(p)])

    def crecer(ids, residual, nivel):
        total = float(residual[ids].sum())
        hoja = {"value": total / len(ids)}
        if nivel == depth or len(ids) < 2 * min_leaf:
            return hoja
        mejor, ganancia_max = None, 1e-12
        for j in range(p):
            m = len(cortes[j])
            cnt = np.bincount(binX[ids, j], minlength=m + 1)
            sums = np.bincount(binX[ids, j], weights=residual[ids], minlength=m + 1)
            cuenta, suma = 0, 0.
            for k in range(m):
                cuenta += int(cnt[k]); suma += float(sums[k])
                derecha = len(ids) - cuenta
                if cuenta < min_leaf or derecha < min_leaf:
                    continue
                ganancia = suma**2/cuenta + (total-suma)**2/derecha - total**2/len(ids)
                if ganancia > ganancia_max:
                    ganancia_max = ganancia
                    mejor = (j, float(cortes[j][k]))
        if mejor is None:
            return hoja
        j, umbral = mejor
        lado = X[ids, j] <= umbral
        return {"feature": j, "threshold": umbral,
                "left": crecer(ids[lado], residual, nivel+1),
                "right": crecer(ids[~lado], residual, nivel+1)}

    for _ in range(rounds):
        arbol = crecer(np.arange(n), z-pred, 0)
        arboles.append(arbol)
        pred += rate * valor_arbol(arbol, X)
    return {"kind": "boost_log1p", "base": base, "rate": rate, "trees": arboles}

CANDIDATOS = [
    {"id": "mediana_por_poblacion", "family": "population_median", "enriched": False},
    {"id": "poisson_demografia", "family": "poisson", "enriched": False},
    {"id": "poisson_capacidades", "family": "poisson", "enriched": True},
    {"id": "boosting_demografia", "family": "boosting", "enriched": False},
    {"id": "boosting_capacidades", "family": "boosting", "enriched": True},
]

def entrenar(frame, config):
    y = frame["urgencias"].to_numpy(dtype=float)
    assert np.isfinite(y).all() and (y > 0).all()
    if config["family"] == "population_median":
        cortes = np.quantile(frame["population_2026"], [.2, .4, .6, .8])
        grupo = (frame["population_2026"].to_numpy()[:, None] > cortes).sum(axis=1)
        medianas = np.array([np.median(y[grupo == j]) if np.any(grupo == j) else np.median(y)
                             for j in range(5)])
        return {"config": config, "model": {"kind": "population_median",
                                           "thresholds": cortes, "medians": medianas}}
    pre = ajustar_preproceso(frame, config["enriched"])
    X = transformar(frame, pre, config["enriched"])
    modelo = (ajustar_poisson(X, y) if config["family"] == "poisson"
              else ajustar_boosting(X, y))
    if modelo.get("converged") is False:
        warnings.warn("Poisson no declaró convergencia: revisar antes de presentar.")
    return {"config": config, "scaler": pre, "model": modelo}

def predecir(ajuste, frame, piso=True):
    modelo = ajuste["model"]
    if modelo["kind"] == "population_median":
        grupo = (frame["population_2026"].to_numpy()[:, None] > modelo["thresholds"]).sum(axis=1)
        pred = modelo["medians"][grupo]
    else:
        X = transformar(frame, ajuste["scaler"], ajuste["config"]["enriched"])
        if modelo["kind"] == "poisson":
            pred = np.exp(modelo["beta"][0] + X @ modelo["beta"][1:])
        else:
            log_pred = np.full(len(X), modelo["base"])
            for arbol in modelo["trees"]:
                log_pred += modelo["rate"] * valor_arbol(arbol, X)
            pred = np.expm1(log_pred)
    return np.maximum(1., pred) if piso else pred


## 8. Métricas: qué se elige y qué se informa
La selección final minimiza el error logarítmico absoluto medio:

\[
E_{\log}=\frac1n\sum_i|\log(\widehat B_i)-\log(B_i)|.
\]

Equivocarse por un factor 2 pesa igual en un municipio pequeño o grande. Como usamos daño/base, un error multiplicativo en la base se transmite al cociente cuando el daño es positivo. **Eso no garantiza conservar el ranking**, porque un máximo también puede cambiar.

Se informan además MAE, RMSE, R² predictivo y error porcentual mediano. R² aquí es \(1-\mathrm{SSE}/\mathrm{SST}\), no la correlación al cuadrado.

**Transparencia:** el primer ensayo se seleccionó por MAE nacional; después de explorar resultados se cambió al error proporcional. Esta es validación exploratoria, no una confirmación externa intacta. El archivo `ensayo_inicial_mae.json` conserva el primer ensayo.


In [ ]:
def metricas(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    assert len(y) == len(p) and len(y) > 0
    assert np.isfinite(y).all() and np.isfinite(p).all() and (y > 0).all() and (p > 0).all()
    ae = np.abs(p-y)
    sst = np.sum((y-y.mean())**2)
    return {"n": len(y), "mae": float(ae.mean()),
            "rmse": float(np.sqrt(np.mean((y-p)**2))),
            "r2": float(1-np.sum((y-p)**2)/sst) if sst > 0 else np.nan,
            "mean_absolute_log_error": float(np.abs(np.log(np.maximum(p, 1))-np.log(y)).mean()),
            "median_absolute_percentage_error": float(np.median(ae/y*100)),
            "within_one_fraction": float(np.mean(ae <= 1))}

assert np.isclose(metricas([1, 100], [2, 200])["mean_absolute_log_error"], np.log(2))
assert metricas([1, 2, 3], [1, 2, 3])["r2"] == 1
assert metricas([1, 2, 3], [3, 2, 1])["r2"] < 0
print("Comprobaciones de métricas: correctas.")


## 9. Separar departamentos completos
Se distribuyen departamentos entre cinco grupos equilibrados por número de municipios observados. Un departamento no aparece simultáneamente en entrenamiento y evaluación.

En cada grupo exterior:
1. Reservamos sus departamentos.
2. Con los restantes hacemos tres particiones internas, también por departamento, para elegir entre los cinco candidatos.
3. Reentrenamos el elegido sin los departamentos reservados.
4. Predecimos sólo esos departamentos.

Esta separación reduce la dependencia geográfica. No resuelve por sí sola el posible sesgo de que los municipios sin registro sean distintos de los municipios observados.


In [ ]:
def particiones(frame, k):
    conteos = Counter(frame["department"])
    if len(conteos) < k:
        raise ValueError("No hay suficientes departamentos.")
    grupos = [{"n": 0, "departments": []} for _ in range(k)]
    for dep, n in sorted(conteos.items(), key=lambda x: (-x[1], x[0])):
        j = min(range(k), key=lambda i: grupos[i]["n"])
        grupos[j]["n"] += n
        grupos[j]["departments"].append(dep)
    return grupos

externos = particiones(observados, 5)
assert len(set(d for g in externos for d in g["departments"])) == observados["department"].nunique()
for nuevo, guardado in zip(externos, historico["folds"]):
    assert nuevo["departments"] == guardado["held_out_departments"]
display(pd.DataFrame([
    {"grupo": i+1, "evaluación": g["n"], "entrenamiento": len(observados)-g["n"],
     "departamentos reservados": ", ".join(g["departments"])}
    for i, g in enumerate(externos)
]))

def seleccionar(frame, k=3):
    resultados = []
    for config in CANDIDATOS:
        reales, predichos = [], []
        for g in particiones(frame, k):
            mascara = frame["department"].isin(g["departments"])
            train, test = frame.loc[~mascara], frame.loc[mascara]
            assert set(train["department"]).isdisjoint(test["department"])
            ajuste = entrenar(train, config)
            reales.extend(test["urgencias"].to_numpy())
            predichos.extend(predecir(ajuste, test))
        resultados.append({"id": config["id"], **metricas(reales, predichos)})
    tabla = pd.DataFrame(resultados).sort_values("mean_absolute_log_error", kind="stable")
    ganador = next(c for c in CANDIDATOS if c["id"] == tabla.iloc[0]["id"])
    return ganador, tabla


## 10. Ejecutar la validación anidada
Esta celda vuelve a entrenar los candidatos. **No copia las predicciones del CSV.** Puede tardar varios minutos; muestra progreso por grupo.

No se utiliza aquí un modelo entrenado previamente con todos los municipios. Las etiquetas del grupo reservado quedan fuera del entrenamiento y de la selección.


In [ ]:
filas_oof, decisiones, tablas_internas = [], [], []
for i, g in enumerate(externos, start=1):
    mascara = observados["department"].isin(g["departments"])
    train, test = observados.loc[~mascara], observados.loc[mascara]
    ganador, tabla = seleccionar(train, k=3)
    tablas_internas.append(tabla.assign(grupo_externo=i))
    ajuste = entrenar(train, ganador)
    pred = predecir(ajuste, test)
    salida = test[["code", "municipality", "department", "population_2026", "urgencias"]].copy()
    salida = salida.rename(columns={"urgencias": "observado"})
    salida["predicho"] = pred
    salida["grupo"] = i
    salida["modelo"] = ganador["id"]
    filas_oof.append(salida)
    decisiones.append({"grupo": i, "modelo": ganador["id"], "n_train": len(train),
                       "n_test": len(test), "convergió": ajuste["model"].get("converged")})
    print(f"Grupo {i}/5: {ganador['id']}; {len(test)} predicciones fuera de muestra.", flush=True)

oof = pd.concat(filas_oof, ignore_index=True)
assert len(oof) == 901 and oof["code"].is_unique
display(pd.DataFrame(decisiones))
display(pd.concat(tablas_internas, ignore_index=True)[
    ["grupo_externo", "id", "mean_absolute_log_error", "mae", "r2"]])


## 11. Contrastar con los resultados históricos del repositorio
Los resultados de referencia son MAE nacional ≈1,24 y R²≈0,787; en municipios de menos de 50.000 habitantes, MAE≈0,41 y R²≈0,343.

La traducción NumPy puede introducir pequeñas diferencias de punto flotante. Se comprueban las predicciones municipio a municipio y el modelo seleccionado. Si una comprobación falla, **no se afirma reproducción exacta**: se debe revisar antes de presentar.


In [ ]:
arch = validacion_archivada.loc[validacion_archivada["denominador"].eq("urgencias")].copy()
arch = arch.rename(columns={"codigo": "code",
                            "predicho_sin_ver_departamento": "predicho_archivado"})
comparacion_oof = oof.merge(arch[["code", "predicho_archivado", "modelo"]],
                           on="code", how="inner", validate="one_to_one", suffixes=("", "_archivado"))
assert len(comparacion_oof) == 901
assert comparacion_oof["modelo"].eq(comparacion_oof["modelo_archivado"]).all()
np.testing.assert_allclose(comparacion_oof["predicho"], comparacion_oof["predicho_archivado"],
                           rtol=1e-5, atol=1e-5)
print("Máxima diferencia absoluta Python frente al registro JavaScript:",
      float(np.max(np.abs(comparacion_oof["predicho"]-comparacion_oof["predicho_archivado"]))))

resumen_metricas = pd.DataFrame([
    {"grupo": "Nacional", **metricas(oof["observado"], oof["predicho"])},
    {"grupo": "Menos de 50.000 habitantes",
     **metricas(oof.loc[oof["population_2026"].lt(50000), "observado"],
                oof.loc[oof["population_2026"].lt(50000), "predicho"])},
])
display(resumen_metricas)


## 12. Gráficos para explicar la validación
**Izquierda:** cada punto es un municipio. La diagonal significa predicción perfecta; arriba se sobreestima y abajo se subestima. Los dos ejes se muestran en escala logarítmica para ver municipios pequeños y grandes, pero las etiquetas conservan consultorios.

**Derecha:** predicho menos observado. Cero es acierto; positivo es sobreestimación. No confundir esta dispersión con un intervalo de confianza.

R² nacional alto puede estar influido por municipios grandes; por eso se muestran métricas separadas por tamaño.


In [ ]:
oof["error"] = oof["predicho"] - oof["observado"]
oof["error_absoluto"] = oof["error"].abs()
oof["error_porcentual"] = 100 * oof["error"] / oof["observado"]

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].scatter(oof["observado"], oof["predicho"], alpha=.45, s=24, color=AZUL)
lo = min(oof["observado"].min(), oof["predicho"].min())
hi = max(oof["observado"].max(), oof["predicho"].max())
axs[0].plot([lo, hi], [lo, hi], "--", color=NARANJA, label="Predicción perfecta")
axs[0].set(xscale="log", yscale="log", xlabel="Consultorios observados (REPS 2022)",
           ylabel="Consultorios predichos fuera de muestra", title="Real frente a predicho")
axs[0].legend()
axs[1].scatter(oof["observado"], oof["error"], alpha=.45, s=24, color=AZUL)
axs[1].axhline(0, color=NARANJA, linestyle="--")
axs[1].set(xscale="log", xlabel="Consultorios observados", ylabel="Predicho − observado",
           title="Error con signo, en consultorios")
plt.tight_layout(); plt.show()

limites = [0, 5000, 10000, 20000, 50000, 100000, np.inf]
etiquetas = ["<5 mil", "5–10 mil", "10–20 mil", "20–50 mil", "50–100 mil", "≥100 mil"]
oof["grupo_poblacion"] = pd.cut(oof["population_2026"], limites, labels=etiquetas, right=False)
por_tamano = pd.DataFrame([
    {"poblacion": str(nombre), **metricas(g["observado"], g["predicho"])}
    for nombre, g in oof.groupby("grupo_poblacion", observed=True)
])
fig, ax = plt.subplots()
ax.bar(por_tamano["poblacion"], por_tamano["mae"], color=AZUL)
ax.set(xlabel="Población proyectada 2026", ylabel="MAE, en consultorios",
       title="El error no es igual en todos los tamaños municipales")
plt.tight_layout(); plt.show()
display(por_tamano)


## 13. Ejemplos municipales y casos difíciles
Estos números son predicciones **fuera de muestra**, no el ajuste sobre datos que el modelo ya vio. Pereira, Trujillo y San José del Palmar tienen etiqueta de urgencias con la que comparar. Atrato no tiene una etiqueta observada de urgencias en esta base: aparecerá después como estimación, no como validación.

Los diez mayores errores se muestran para no seleccionar únicamente ejemplos favorables.


In [ ]:
CODIGOS_EJEMPLO = ["66001", "76828", "27660", "27050"]
columnas_ejemplo = ["code", "municipality", "department", "observado", "predicho",
                    "error", "error_porcentual", "grupo"]
display(oof.loc[oof["code"].isin(CODIGOS_EJEMPLO), columnas_ejemplo])
display(oof.nlargest(10, "error_absoluto")[columnas_ejemplo])


## 14. Elegir y ajustar el modelo final con todos los observados
Una vez evaluado el procedimiento fuera de muestra, se selecciona el candidato con cinco particiones departamentales sobre los 901 observados y se ajusta con todos ellos para estimar faltantes.

Esta tabla de selección no sustituye la evaluación exterior anterior. Los coeficientes describen asociaciones predictivas, no efectos causales de la población, la pobreza o la inversión.


In [ ]:
config_final, tabla_final = seleccionar(observados, k=5)
display(tabla_final[["id", "mean_absolute_log_error", "mae", "r2"]])
assert config_final["id"] == historico["final_selection"]["config"]["id"] == "poisson_capacidades"
ajuste_final = entrenar(observados, config_final)
assert ajuste_final["model"]["converged"]

np.testing.assert_allclose(
    ajuste_final["model"]["beta"], historico["model"]["model"]["beta"],
    rtol=1e-5, atol=1e-5
)
pre_final = ajuste_final["scaler"]
coeficientes = pd.DataFrame({
    "variable": ["Intercepto"] + NOMBRES_X,
    "coeficiente": ajuste_final["model"]["beta"],
    "media_entrenamiento": [np.nan] + list(pre_final["avg"]),
    "desviacion_entrenamiento": [np.nan] + list(pre_final["sd"]),
})
display(coeficientes)
print("Fórmula: B_estimado = max(1, exp(intercepto + suma(coeficiente_j * z_j))).")


## 15. Contraste opcional con una implementación independiente
Se ajusta `PoissonRegressor` de scikit-learn sobre exactamente las mismas variables estandarizadas y con \(\alpha=1\). Esta celda comprueba si ambos optimizadores convergen a predicciones compatibles.

No se presenta este contraste como una prueba ya ejecutada al escribir el cuaderno. El mensaje que aparezca al ejecutar la celda es el resultado real de la comprobación.


In [ ]:
contraste_sklearn = {"estado": "no ejecutado"}
try:
    import sklearn
    from sklearn.linear_model import PoissonRegressor
except ImportError:
    print("Contraste omitido: instala scikit-learn si deseas ejecutarlo.")
else:
    X_final = transformar(observados, pre_final, True)
    modelo_sklearn = PoissonRegressor(alpha=1., max_iter=5000, tol=1e-9)
    modelo_sklearn.fit(X_final, observados["urgencias"].to_numpy())
    propio = predecir(ajuste_final, observados, piso=False)
    independiente = modelo_sklearn.predict(X_final)
    diferencia = float(np.max(np.abs(propio-independiente)))
    np.testing.assert_allclose(propio, independiente, rtol=1e-4, atol=1e-4)
    contraste_sklearn = {"estado": "predicciones compatibles", "version": sklearn.__version__,
                        "max_diferencia_absoluta": diferencia}
    display(pd.Series(contraste_sklearn))


## 16. Estimar sólo faltantes con soporte mínimo
Para publicar una estimación experimental se exige:

- IPM observado.
- Predictores dentro del rango de entrenamiento.
- Al menos 30 municipios observados con el mismo patrón de presencia/ausencia de las otras capacidades.

Estos controles reducen extrapolaciones evidentes, pero **no son una garantía validada de precisión**. No prueban existencia del servicio. El esperado histórico es 203 estimaciones elegibles y 18 abstenciones entre 221 faltantes. No se redondean estimaciones decimales.


In [ ]:
def patron(frame):
    return [(pd.isna(a), pd.isna(b)) for a, b in zip(frame["consulta_externa"], frame["camas_generales"])]

conteos_patron = Counter(patron(observados))
X_train_crudo = variables(observados)
min_train = np.array([np.min(c[np.isfinite(c)]) for c in X_train_crudo.T])
max_train = np.array([np.max(c[np.isfinite(c)]) for c in X_train_crudo.T])
X_missing = variables(faltantes)
fuera = np.isfinite(X_missing) & ((X_missing < min_train) | (X_missing > max_train))

estimaciones = faltantes[["code", "municipality", "department", "ipm_2018"]].copy()
estimaciones["prediccion_sin_piso"] = predecir(ajuste_final, faltantes, piso=False)
estimaciones["prediccion_con_piso"] = predecir(ajuste_final, faltantes)
estimaciones["n_mismo_patron"] = [conteos_patron[p] for p in patron(faltantes)]
estimaciones["fuera_de_rango"] = fuera.any(axis=1)
estimaciones["soporte"] = (estimaciones["n_mismo_patron"].ge(30)
                           & ~estimaciones["fuera_de_rango"]
                           & estimaciones["ipm_2018"].notna())
estimaciones["estimacion_publicable"] = estimaciones["prediccion_con_piso"].where(estimaciones["soporte"])
assert estimaciones["soporte"].sum() == 203
assert (~estimaciones["soporte"]).sum() == 18

guardadas = pd.DataFrame(historico["predictions"])
chequeo_estimaciones = estimaciones.merge(
    guardadas[["code", "predicted", "exploratory_supported"]], on="code", validate="one_to_one")
np.testing.assert_allclose(chequeo_estimaciones["prediccion_con_piso"],
                           chequeo_estimaciones["predicted"], rtol=1e-5, atol=1e-5)
assert chequeo_estimaciones["soporte"].eq(chequeo_estimaciones["exploratory_supported"]).all()
display(estimaciones.loc[estimaciones["code"].isin(CODIGOS_EJEMPLO)])
display(estimaciones.loc[~estimaciones["soporte"]])


## 17. Mostrar el efecto del piso 1, no ocultarlo
El modelo puede producir una media menor que uno. El escenario publicado usa \(\max(1,\mu)\), porque el ensayo sólo aprendió de registros positivos. **Esto no permite concluir que los municipios sin registro tengan al menos un consultorio.**

Una media estimada de 0,6 tampoco es prueba de que existan «0,6 consultorios»: es una predicción estadística, no un conteo observado.

En el experimento original, 192 de las 221 predicciones brutas para faltantes terminan en 1. La falta de ejemplos de cero real es una limitación importante para aplicar el modelo.


In [ ]:
n_piso = int(estimaciones["prediccion_con_piso"].eq(1).sum())
assert n_piso == 192
print(f"{n_piso}/{len(estimaciones)} predicciones brutas quedan en el piso 1.")
fig, ax = plt.subplots()
ax.scatter(estimaciones["prediccion_sin_piso"], estimaciones["prediccion_con_piso"],
           c=np.where(estimaciones["soporte"], AZUL, NARANJA), alpha=.55)
max_graf = max(1., estimaciones["prediccion_con_piso"].max())
ax.plot([0, max_graf], [0, max_graf], "--", color="gray", label="Sin recorte")
ax.axhline(1, color=NARANJA, linewidth=1, label="Piso del escenario")
ax.set(xlabel="Predicción antes del piso", ylabel="Predicción después del piso",
       title="Efecto explícito de la restricción a capacidad positiva")
ax.legend(); plt.tight_layout(); plt.show()


## 18. Cargar los daños que realmente utiliza el tablero
El `index.html` del commit congelado contiene el inventario depurado, sus códigos municipales y las reglas territoriales ya aplicadas. Leemos únicamente el objeto JSON `DATA`: **no ejecutamos JavaScript del HTML**.

Esto evita inventar un daño para el ejemplo o resolver nombres municipales con coincidencias aproximadas. Se usa exclusivamente el indicador `pnud_csalud`, fuente `PNUD`, unidad `Número`, del mismo corte. No se rellenan sus ausencias con 3iS.

El ámbito puede ser `"decree"` (departamentos del decreto) o `"all"` (todo el inventario). El máximo se calcula de nuevo para ese ámbito.


In [ ]:
html_tablero = leer_version("index.html").decode("utf-8")
marcador = re.search(r"\bconst\s+DATA\s*=\s*", html_tablero)
if marcador is None:
    raise ValueError("No se encontró el JSON DATA del tablero.")
payload, _ = json.JSONDecoder().raw_decode(html_tablero[marcador.end():])
CAPTURA = "2026-09-11"
AMBITO = "decree"  # Cambiar a "all" para todo el inventario.
assert AMBITO in {"decree", "all"} and payload["latest"] == CAPTURA

inventario = pd.DataFrame(payload["rows"])
actual = inventario.loc[inventario["date"].eq(CAPTURA)].copy()
departamentos_decreto = set(actual.loc[
    actual["id"].eq("en_decreto_1171") & actual["f"].eq("Decreto1171") & actual["v"].eq(1), "d"
])
municipales = actual.loc[actual["lv"].eq("municipal")].copy()
if AMBITO == "decree":
    municipales = municipales.loc[municipales["d"].isin(departamentos_decreto)]
territorios = municipales[["geo", "code", "m", "d"]].drop_duplicates("geo")

salud_pnud = municipales.loc[
    municipales["id"].eq("pnud_csalud") & municipales["f"].eq("PNUD")
    & municipales["u"].eq("Número"), ["geo", "v"]
].rename(columns={"v": "afectados"})
assert salud_pnud["geo"].is_unique
assert salud_pnud["afectados"].ge(0).all()

# Verificar que el registro municipal de entrenamiento coincide con los denominadores del tablero.
den_tab = pd.DataFrame(payload["denominators"]["rows"])
den_observados = den_tab.loc[
    den_tab["kind"].eq("consultorios_urgencias_reps")
    & den_tab["status"].eq("verified_historical"), ["code", "value"]
]
assert len(den_observados) == 901 and den_observados["code"].is_unique
ver_capacidad = observados.merge(den_observados, on="code", validate="one_to_one")
assert len(ver_capacidad) == 901
np.testing.assert_allclose(ver_capacidad["urgencias"], ver_capacidad["value"], rtol=0, atol=0)
print(f"{len(territorios)} territorios de referencia; ámbito={AMBITO}; captura={CAPTURA}.")
display(salud_pnud.head())


## 19. Del denominador al puntaje de Salud
\[
q_m = \frac{A_m}{B_m},\qquad S_m=100\frac{q_m}{\max_{j\in R}(q_j)}.
\]

- \(A_m\): centros afectados PNUD; falta → no se calcula.
- \(B_m\): consultorios observados; sólo cuando faltan, estimación elegible.
- \(R\): municipios comparables del ámbito y corte seleccionado.
- Daño cero explícito con denominador utilizable → presión y puntaje cero.
- Sin ambos componentes no hay dato relativo.

**No es porcentaje de consultorios destruidos:** el numerador son centros/puntos afectados y el denominador consultorios. Es presión relativa sobre una capacidad distinta.

Se muestran dos escenarios: urgencias observadas y urgencias con ML. No se reemplazan los registros existentes.


In [ ]:
panel = territorios.merge(salud_pnud, on="geo", how="left", validate="one_to_one")
panel = panel.merge(df[["code", "urgencias", "ipm_2018"]], on="code",
                    how="left", validate="many_to_one")
panel = panel.merge(estimaciones[["code", "estimacion_publicable"]], on="code",
                    how="left", validate="many_to_one")
panel["base_observada"] = panel["urgencias"]
panel["base_escenario"] = panel["base_observada"].fillna(panel["estimacion_publicable"])
panel["base_es_ML"] = panel["base_observada"].isna() & panel["estimacion_publicable"].notna()
anclas = {}
for sufijo, base in [("observado", "base_observada"), ("ml", "base_escenario")]:
    panel[f"presion_{sufijo}"] = panel["afectados"] / panel[base]
    maximo = panel[f"presion_{sufijo}"].max()
    if not np.isfinite(maximo) or maximo <= 0:
        raise ValueError("No hay un máximo positivo para normalizar este ámbito.")
    anclas[sufijo] = float(maximo)
    panel[f"salud_{sufijo}"] = (100*panel[f"presion_{sufijo}"]/maximo).clip(0, 100)

resumen_aplicacion = {
    "territorios": len(panel),
    "Salud calculable sin ML": int(panel["salud_observado"].notna().sum()),
    "Salud calculable con ML": int(panel["salud_ml"].notna().sum()),
    "Salud calculada usando ML": int((panel["base_es_ML"] & panel["salud_ml"].notna()).sum()),
    "máximo sin ML": anclas["observado"], "máximo con ML": anclas["ml"],
}
display(pd.Series(resumen_aplicacion))

auditoria_aplicacion = json.loads(leer_version("experimentos/ml_salud/impacto_aplicacion_urgencias.json"))
reporte = next(r for r in auditoria_aplicacion
               if r["scope"] == AMBITO and r["severity"] == "total" and r["date"] == CAPTURA)
assert resumen_aplicacion["territorios"] == reporte["stats"]["ml"]["reference"]
assert resumen_aplicacion["Salud calculable sin ML"] == reporte["stats"]["urgencias"]["health"]
assert resumen_aplicacion["Salud calculable con ML"] == reporte["stats"]["ml"]["health"]
assert resumen_aplicacion["Salud calculada usando ML"] == reporte["stats"]["ml"]["imputed"]
np.testing.assert_allclose(anclas["ml"], reporte["stats"]["ml"]["anchor"], rtol=1e-5)

# Comparación por municipio: el cálculo nuevo debe corresponder al puntaje de la rama.
salud_guardada = pd.DataFrame([
    {"code": r["code"], "limite_salud_archivado": r["ml"]["health"]} for r in reporte["changes"]
])
comparar_salud = panel.loc[panel["code"].ne("")].merge(salud_guardada, on="code", validate="one_to_one")
# El archivo guarda el límite documentado: un sector sin dato tiene límite 0.
# Sólo para comparar límites se usa fillna(0); panel.salud_ml conserva NaN.
np.testing.assert_allclose(comparar_salud["salud_ml"].fillna(0), comparar_salud["limite_salud_archivado"],
                           rtol=1e-5, atol=1e-5, equal_nan=True)
display(panel.loc[panel["code"].isin(CODIGOS_EJEMPLO),
                  ["code", "m", "afectados", "base_observada", "estimacion_publicable",
                   "base_escenario", "presion_ml", "salud_ml", "base_es_ML"]])


## 20. Atrato: una explicación que se puede presentar
Para el corte congelado, PNUD reporta 9 centros afectados; el escenario ML aporta un denominador de 1 consultorio de urgencias. Con máximo 16, Salud es \(100\times(9/1)/16=56,25\).

El índice global existente da peso \(1/6\) a Salud y luego aplica el factor IPM:
\[
P_m=D_m\frac{1+0,25\,IPM_m/100}{1,25}.
\]
Como el máximo no cambió en este experimento y las otras dimensiones permanecen iguales, podemos aislar el cambio de Salud:
\[
\Delta P_m=\frac{S_m^{ML}-S_m^{sin\,ML}}6
 \frac{1+0,25\,IPM_m/100}{1,25}.
\]

**En el límite documentado del índice**, un campo ausente aporta cero; eso no convierte el daño o la capacidad ausente en cero observado. La siguiente celda utiliza esa convención sólo al calcular la diferencia entre límites del escenario. No reconstruye las otras cinco dimensiones.


In [ ]:
atrato = panel.loc[panel["code"].eq("27050")].iloc[0]
atrato_reporte = next(r for r in reporte["changes"] if r["code"] == "27050")
factor_ipm = (1 + .25*atrato["ipm_2018"]/100) / 1.25
salud_antes = 0. if pd.isna(atrato["salud_observado"]) else atrato["salud_observado"]
delta_p = (atrato["salud_ml"]-salud_antes)/6 * factor_ipm
np.testing.assert_allclose(delta_p, atrato_reporte["delta_score"], rtol=1e-5)
display(pd.Series({
    "Centros afectados PNUD": atrato["afectados"],
    "Consultorios observados": atrato["base_observada"],
    "Consultorios del escenario ML": atrato["base_escenario"],
    "Presión": atrato["presion_ml"],
    "Máximo del ámbito": anclas["ml"],
    "Puntaje Salud": atrato["salud_ml"],
    "IPM 2018": atrato["ipm_2018"],
    "Factor IPM": factor_ipm,
    "Incremento del puntaje global": delta_p,
    "Global urgencias sin ML (archivo)": atrato_reporte["urgencias"]["score"],
    "Global urgencias con ML (archivo)": atrato_reporte["ml"]["score"],
    "Puesto sin ML (archivo)": atrato_reporte["urgencias"]["rank"],
    "Puesto con ML (archivo)": atrato_reporte["ml"]["rank"],
}))
display(Markdown("**Interpretación:** el aumento depende de una base estimada en el piso 1. "
                 "El modelo no confirmó la existencia de un consultorio en Atrato."))


## 21. Cuánto depende del denominador: sensibilidad, no intervalo de confianza
Manteniendo fija el ancla 16 del escenario, cambiamos hipotéticamente la base de Atrato. Las alternativas 1, 2, 3, 5 y 10 **no son registros encontrados**, ni límites estadísticos: sirven para explicar la sensibilidad inversa de daño/base.

Si se modificaran bases de muchos municipios habría que recalcular también el máximo y el ranking.


In [ ]:
bases_hipoteticas = np.array([1., 2., 3., 5., 10.])
salud_hipotetica = np.minimum(100., 100 * (atrato["afectados"]/bases_hipoteticas) / anclas["ml"])
fig, ax = plt.subplots()
ax.plot(bases_hipoteticas, salud_hipotetica, "o-", color=AZUL)
ax.set(xlabel="Consultorios de urgencias hipotéticos — no observaciones",
       ylabel="Puntaje Salud / 100",
       title="Atrato: sensibilidad del puntaje a la base, con máximo fijo")
ax.set_xticks(bases_hipoteticas)
plt.tight_layout(); plt.show()
display(pd.DataFrame({"base_hipotetica": bases_hipoteticas, "salud_con_ancla_fija": salud_hipotetica}))


## 22. Cambios del ranking y la comparación con UNGRD
El archivo de auditoría contiene el cálculo completo de las seis dimensiones para cada escenario. Aquí se lee ese resultado **archivado**; no se presenta como si este cuaderno hubiera recalculado las demás dimensiones.

Se comparan urgencias sin ML y urgencias con ML. El cambio desde consulta externa también altera el denominador; no hay que confundir ambos efectos.

R² con UNGRD es asociación entre índices, **no validación del número de consultorios** ni evidencia de causalidad. El modelo no fue entrenado para maximizarlo.


In [ ]:
ranking_archivado = pd.DataFrame([
    {"municipio": r["m"], "departamento": r["d"],
     "puntaje_sin_ML": r["urgencias"]["score"], "puntaje_con_ML": r["ml"]["score"],
     "puesto_sin_ML": r["urgencias"]["rank"], "puesto_con_ML": r["ml"]["rank"],
     "cambio_puntaje": r["delta_score"], "campos_estimados": r["ml"]["estimated"]}
    for r in reporte["changes"]
])
display(ranking_archivado.sort_values("puesto_con_ML").head(15))
display(pd.DataFrame([
    {"escenario": nombre, "pares_UNGRD": reporte["stats"][clave]["n"],
     "R2_con_UNGRD": reporte["stats"][clave]["regression"]["r2"],
     "Spearman_con_UNGRD": reporte["stats"][clave]["rho"]}
    for clave, nombre in [("consulta", "Consulta externa observada"),
                          ("urgencias", "Urgencias observadas"), ("ml", "Urgencias con ML")]
]))


## 23. Qué decir y qué no decir al presentar

**Una explicación defendible:**
> «Estimamos capacidad histórica de urgencias sólo donde faltaba el registro. Validamos ocultando departamentos completos y conservamos los datos observados. El resultado permite explorar un escenario de presión sanitaria, con las estimaciones identificadas».

**Limitaciones que deben acompañarlo:**
- Capacidad 2022, IPM 2018 y población proyectada 2026 no son un inventario contemporáneo.
- No hay ejemplos de capacidad realmente cero. El piso 1 es una restricción del escenario, no una verificación del servicio.
- Validación sobre observados no garantiza el mismo rendimiento en faltantes.
- El criterio de selección se refinó después de explorar resultados; no existe validación externa intacta.
- Puntos afectados y consultorios no son unidades equivalentes: no se obtiene porcentaje de consultorios destruidos.
- Usar IPM como predictor de capacidad y mantenerlo en el ajuste global introduce una influencia indirecta adicional de pobreza.
- Los límites del tablero por variables faltantes **no incluyen error de ML** ni cambios temporales; no son intervalos de confianza.
- El máximo de referencia transmite errores: si cambia un máximo, pueden cambiar puntajes de otros municipios.
- Un mejor R² frente a UNGRD no demuestra que el denominador estimado sea correcto.

**No afirmar:** «Atrato tiene un consultorio», «el modelo verificó el daño», «0,787 significa 78,7% de predicciones correctas», o «urgencias es el denominador verdadero».

**Preguntas para el auditorio:** ¿qué decisión cambia al usar el escenario? ¿El cambio depende de una base observada o estimada? ¿Sería defendible si esa estimación duplicara su valor?


## 24. Exportación opcional, sin tocar el tablero
Por defecto no se escribe ningún archivo. Si se activa la exportación, se crea una carpeta nueva de resultados del notebook; no se sobrescribe una carpeta existente. Esto no publica estimaciones como registros oficiales ni regenera `index.html`.


In [ ]:
EXPORTAR = False
if EXPORTAR:
    from datetime import datetime
    destino = Path.cwd() / ("salidas_notebook_salud_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    destino.mkdir(exist_ok=False)
    oof.to_csv(destino / "validacion_python_urgencias.csv", index=False)
    estimaciones.to_csv(destino / "estimaciones_python_urgencias.csv", index=False)
    panel.to_csv(destino / "salud_escenario.csv", index=False)
    coeficientes.to_csv(destino / "coeficientes_poisson.csv", index=False)
    pd.DataFrame(trazabilidad).to_csv(destino / "trazabilidad.csv", index=False)
    print(f"Resultados guardados en: {destino.resolve()}")
else:
    print("Sin escritura de archivos. No se ha modificado el tablero.")


## Referencias y archivos para acompañar la presentación

- [Datos municipales del experimento](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/experimentos/ml_salud/entrada.json).
- [Método original y resultados](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/docs/ML_SALUD.md).
- [Código JavaScript original](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/scripts/ml_salud.cjs).
- [Aplicación experimental al tablero](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/docs/ML_URGENCIAS_APLICADO.md).
- [REPS — capacidad instalada, fuente oficial](https://www.datos.gov.co/Salud-y-Protecci-n-Social/Relaci-n-de-IPS-p-blicas-y-privadas-seg-n-el-nivel/s2ru-bqt6/about_data).
- [DANE — proyecciones de población](https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/proyecciones-de-poblacion).
- [DANE — IPM municipal censal 2018](https://www.dane.gov.co/files/investigaciones/condiciones_vida/pobreza/2018/informacion-censal/anexo-censal-pobreza-municipal-2018.xlsx).
- [NumPy: solución del sistema de Newton](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).
- [scikit-learn: PoissonRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.PoissonRegressor.html).

**Trazabilidad de la autoría:** notebook preparado el 15 de septiembre de 2026. Python no pudo ejecutarse en el entorno de creación; no se incluyen salidas aparentando una ejecución. Los datos y resultados históricos se verificaron contra el repositorio, y las celdas contienen controles para la ejecución del usuario.
